In [179]:
import dash
from dash import dcc, html
from dash.dependencies import Input, Output
import plotly.express as px
import pandas as pd
import numpy as np
import textwrap
from deep_translator import GoogleTranslator

df_missoes = pd.read_csv('thor_wwii_data_clean.csv', low_memory=False)
df_avioes = pd.read_csv('thor_wwii_aircraft_gloss.csv')
df_armas = pd.read_csv('thor_wwii_weapon_gloss.csv')
df_intel = pd.read_csv('thor_aircraft_intel.csv')



#  PIPELINE DE DADOS (INTEGRAÇÃO, LIMPEZA E TRANSFORMAÇÃO)


In [180]:
#Merge com csv's
df_completo = pd.merge(df_missoes, df_avioes[['aircraft', 'aircraft_type', 'full_name']], how='left', left_on='mds', right_on='aircraft')
df_completo = pd.merge(df_completo, df_armas[['weapon_name', 'weapon_class']], how='left', left_on='type_of_frag', right_on='weapon_name')
df_completo = pd.merge(df_completo, df_intel[['full_name', 'lore_historico']], how='left', on='full_name')

In [181]:
#Adicionando coisas nos nulos
df_completo['full_name'] = df_completo['full_name'].fillna('Aeronave Não Identificada')
df_completo['lore_historico'] = df_completo['lore_historico'].fillna("Arquivo classificado ou não disponível.")
df_completo['theater'] = df_completo['theater'].fillna('Não Informado')
df_completo['aircraft_type'] = df_completo['aircraft_type'].fillna('Outros')
df_completo['weapon_class'] = df_completo['weapon_class'].fillna('Não Identificado')

In [ ]:
# Datas
df_completo['msndate'] = pd.to_datetime(df_completo['msndate'], errors='coerce')
df_completo['Ano'] = df_completo['msndate'].dt.year
df_completo['Mes'] = df_completo['msndate'].dt.month

# Pesos e Intensidade
df_completo['total_tons'] = pd.to_numeric(df_completo['total_tons'], errors='coerce').fillna(0)
df_completo['Intensidade_Ataque'] = pd.cut(
    df_completo['total_tons'],
    bins=[-1, 10, 50, float('inf')],
    labels=['Leve', 'Médio', 'Pesado']
)

df_completo['aircraft_type'] = df_completo['aircraft_type'].astype(str).str.strip()

dicionario_colunas = {
    'wwii_id': 'ID_Missao',
    'msndate': 'Data_Missao',
    'theater': 'Teatro_Operacoes',
    'country_flying_mission': 'Pais_Atacante',
    'tgt_country': 'Pais_Alvo',
    'tgt_location': 'Localizacao_Alvo',
    'tgt_type': 'Tipo_Alvo',
    'aircraft': 'Sigla_Aeronave',
    'aircraft_type': 'Categoria_Aeronave',
    'full_name': 'Aeronave_Modelo',
    'weapon_name': 'Nome_Armamento',
    'weapon_class': 'Classe_Armamento',
    'total_tons': 'Total_Toneladas',
    'latitude': 'Latitude',
    'longitude': 'Longitude'
}
df_completo.rename(columns=dicionario_colunas, inplace=True)

dicionario_teatros = {
    'ETO': 'Teatro Europeu (ETO)',
    'MTO': 'Mediterrâneo (MTO)',
    'PTO': 'Pacífico (PTO)',
    'CBI': 'China-Birmânia-Índia (CBI)',
    'EAST AFRICA': 'África Oriental',
    'UNKNOWN': 'Desconhecido'
}

def traduzir_avioes_bruto(texto):
    texto = str(texto).upper() 

    if 'BOSTON' in texto: return 'Bombardeiro Leve / Caça Noturno'
    if 'GROUND ATTACK' in texto: return 'Ataque ao Solo / Bomb. de Mergulho'
    if 'NIGHT FIGHTER' in texto: return 'Caça Noturno'
    if 'FIGHTER BOMBER' in texto: return 'Caça-Bombardeiro'
    if 'HEAVY BOMBER' in texto: return 'Bombardeiro Pesado'
    if 'MEDIUM BOMBER' in texto: return 'Bombardeiro Médio'
    if 'LIGHT BOMBER' in texto: return 'Bombardeiro Leve'
    if 'DIVE BOMBER' in texto: return 'Bombardeiro de Mergulho'
    if 'FIGHTER' in texto: return 'Caça'
    if 'RECONNAISSANCE' in texto: return 'Reconhecimento'
    if 'TRANSPORT' in texto: return 'Transporte'
    if 'OBSERVATION' in texto: return 'Observação'
    if 'PATROL' in texto: return 'Patrulha'
    if 'UTILITY' in texto: return 'Utilitário'
    if 'TRAINER' in texto: return 'Treinamento'
    
    return 'Não Identificado'

dicionario_paises = {
    'ITALY': 'Itália',
    'GERMANY': 'Alemanha',
    'FRANCE': 'França',
    'BURMA': 'Birmânia',
    'NEW GUINEA': 'Nova Guiné',
    'PHILIPPINE ISLANDS': 'Filipinas',
    'CHINA': 'China',
    'ROMANIA': 'Romênia',
    'YUGOSLAVIA': 'Iugoslávia',
    'NETHERLANDS': 'Holanda',
    'JAPAN': 'Japão',
    'AUSTRIA': 'Áustria',
    'POLAND': 'Polônia',
    'BELGIUM': 'Bélgica',
    'UNKNOWN': 'Desconhecido'
}

#TRADUZINDO A LORE DOS AVIÕES
tradutor = GoogleTranslator(source='en', target='pt')
textos_unicos = df_completo['lore_historico'].dropna().unique()
dicionario_lore = {}

for texto in textos_unicos:
    # Se o texto já for nulo ou o aviso de não encontrado, pula a tradução
    if pd.isna(texto) or texto == 'Nenhum registro encontrado.' or str(texto) == 'nan':
        dicionario_lore[texto] = 'Nenhum registro encontrado.'
    else:
        try:
            texto_traduzido = tradutor.translate(str(texto))
            dicionario_lore[texto] = '<br>'.join(textwrap.wrap(texto_traduzido, width=60))
        except:
            dicionario_lore[texto] = '<br>'.join(textwrap.wrap(str(texto), width=60))

def quebrar_texto(texto):
    if pd.isna(texto) or texto == 'Nenhum registro encontrado.':
        return texto
    # corta o texto a cada 60 caracteres e unir com a tag <br> do HTML
    return '<br>'.join(textwrap.wrap(str(texto), width=60))

df_completo['Teatro_Operacoes'] = df_completo['Teatro_Operacoes'].replace(dicionario_teatros).fillna('Desconhecido')
df_completo['Categoria_Aeronave'] = df_completo['Categoria_Aeronave'].apply(traduzir_avioes_bruto)
df_completo['Pais_Alvo'] = df_completo['Pais_Alvo'].replace(dicionario_paises).fillna('Desconhecido')
df_completo['lore_historico'] = df_completo['lore_historico'].map(dicionario_lore).fillna('Nenhum registro encontrado.')


# INICIALIZAÇÃO DO APP E CONFIGURAÇÃO DO LAYOUT


In [183]:
app = dash.Dash(__name__, title="Dashboard THOR WWII", suppress_callback_exceptions=True)

# Colocando coisas nos valores nulos
df_completo['Categoria_Aeronave'] = df_completo['Categoria_Aeronave'].fillna('Não Identificado')
df_completo['Tipo_Alvo'] = df_completo['Tipo_Alvo'].fillna('Não Identificado')


app.layout = html.Div(id='main-container', children=[
    html.Header(style={'borderBottom': '1px solid #777', 'paddingBottom': '15px', 'marginBottom': '20px', 'display': 'flex', 'justifyContent': 'space-between', 'alignItems': 'center'}, children=[
        html.Div([
            #Titulo e sub-titulo
            html.H1("Histórico de Operações Aéreas — Segunda Guerra Mundial", style={'margin': '0 0 5px 0'}),
            html.P("Análise descritiva orientada a insights baseada no framework THOR", style={'margin': '0', 'opacity': '0.7'})
        ]),
        # O Botão para mudar de cor
        dcc.RadioItems(
            id='tema-toggle',
            options=[
                {'label': ' ☀️ ', 'value': 'light'},
                {'label': ' 🌙 ', 'value': 'dark'}
            ],
            value='light',
            inline=True,
            style={'fontWeight': 'bold', 'fontSize': '18px', 'cursor': 'pointer'}
        )
    ]),
    #Nomes dos Dashboards
    dcc.Tabs(id="abas-navegacao", value='aba-executiva', children=[
        dcc.Tab(label='Dashboard 1 — Visão Geral Executiva', value='aba-executiva'),
        dcc.Tab(label='Dashboard 2 — Exploração Interativa', value='aba-exploratoria'),
    ]),
    
    html.Div(id='painel-conteudo')
])


# CRIANDO OS DASHBOARDS


In [184]:
@app.callback(
    Output('main-container', 'style'),
    Input('tema-toggle', 'value')
)
#Muda a cor do fundo
def atualizar_fundo_geral(tema):
    if tema == 'dark':
        return {'backgroundColor': '#111111', 'color': '#ffffff', 'minHeight': '100vh', 'fontFamily': 'sans-serif', 'padding': '20px', 'transition': '0.3s'}
    return {'backgroundColor': '#ffffff', 'color': '#000000', 'minHeight': '100vh', 'fontFamily': 'sans-serif', 'padding': '20px', 'transition': '0.3s'}


# Renderiza as Abas e aplica as cores nos Cards
@app.callback(
    Output('painel-conteudo', 'children'),
    [Input('abas-navegacao', 'value'),
     Input('tema-toggle', 'value')] # O tema entra aqui para pintar os Cards
)
def renderizar_aba(aba_selecionada, tema):
    # Lógica de cores baseada no tema
    bg_card = '#222222' if tema == 'dark' else '#ffffff'
    borda_card = '#444444' if tema == 'dark' else '#eeeeee'
    template_grafico = 'plotly_dark' if tema == 'dark' else 'plotly_white'
    
    if aba_selecionada == 'aba-executiva':
        # DASHBOARD 1: Cálculos com os NOVOS nomes das colunas
        total_missoes = f"{len(df_completo):,}".replace(",", ".")
        
        # ATUALIZADO: 'total_tons' virou 'Total_Toneladas' e 'tgt_country' virou 'Pais_Alvo'
        total_toneladas = f"{int(df_completo['Total_Toneladas'].sum()):,}".replace(",", ".")
        paises_atingidos = df_completo['Pais_Alvo'].nunique()

        # Blindagem do eixo temporal
        df_linha_ano = df_completo.dropna(subset=['Ano']).copy()
        df_linha_ano = df_linha_ano.groupby('Ano')['Total_Toneladas'].sum().reset_index()
        df_linha_ano['Ano'] = df_linha_ano['Ano'].astype(int)

        # Gráfico atualizado para ler 'Total_Toneladas'
        fig_temporal_sintetica = px.line(
            df_linha_ano, x='Ano', y='Total_Toneladas',
            title='Curva de Intensidade Logística: Toneladas Lançadas por Ano (1939-1945)',
            labels={'Total_Toneladas': 'Toneladas de Bombas', 'Ano': 'Ano do Conflito'}
        )
        fig_temporal_sintetica.update_layout(template=template_grafico)
        fig_temporal_sintetica.update_xaxes(dtick=1)
        
        # Indicadores dos cards
        return html.Div(style={'paddingTop': '20px'}, children=[
            html.Div(style={'display': 'flex', 'gap': '20px', 'marginBottom': '30px'}, children=[
                html.Div(style={'flex': '1', 'backgroundColor': bg_card, 'border': f'1px solid {borda_card}', 'padding': '25px', 'borderRadius': '8px'}, children=[
                    html.H4("VOLUME DE MISSÕES", style={'margin': '0 0 10px 0', 'fontSize': '12px', 'opacity': '0.7', 'letterSpacing': '1px'}),
                    html.P(total_missoes, style={'margin': '0', 'fontSize': '32px', 'fontWeight': 'bold'})
                ]),
                html.Div(style={'flex': '1', 'backgroundColor': bg_card, 'border': f'1px solid {borda_card}', 'padding': '25px', 'borderRadius': '8px'}, children=[
                    html.H4("CARGA DESTRUTIVA", style={'margin': '0 0 10px 0', 'fontSize': '12px', 'opacity': '0.7', 'letterSpacing': '1px'}),
                    html.P(f"{total_toneladas} Tons", style={'margin': '0', 'fontSize': '32px', 'fontWeight': 'bold'})
                ]),
                html.Div(style={'flex': '1', 'backgroundColor': bg_card, 'border': f'1px solid {borda_card}', 'padding': '25px', 'borderRadius': '8px'}, children=[
                    html.H4("ALVOS GEOGRÁFICOS (PAÍSES)", style={'margin': '0 0 10px 0', 'fontSize': '12px', 'opacity': '0.7', 'letterSpacing': '1px'}),
                    html.P(paises_atingidos, style={'margin': '0', 'fontSize': '32px', 'fontWeight': 'bold'})
                ]),
            ]),
            
            html.Div(style={'backgroundColor': bg_card, 'border': f'1px solid {borda_card}', 'padding': '20px', 'borderRadius': '8px'}, children=[
                dcc.Graph(figure=fig_temporal_sintetica)
            ])
        ])
        
    elif aba_selecionada == 'aba-exploratoria':
        # DASHBOARD 2:
        lista_teatros = sorted(df_completo['Teatro_Operacoes'].dropna().unique())
        lista_intensidades = sorted(df_completo['Intensidade_Ataque'].dropna().unique())
        
        estilo_filtro = {'color': '#000000'} 
        
        return html.Div(style={'paddingTop': '20px'}, children=[
            # Painel lateral dos filtros(Teatro de Operações e Intensidade do Ataque)
            html.Div(style={'backgroundColor': bg_card, 'border': f'1px solid {borda_card}', 'padding': '20px', 'borderRadius': '8px', 'marginBottom': '20px'}, children=[
                html.H3("Filtros de Segmentação Tática", style={'margin': '0 0 15px 0', 'fontSize': '16px'}),
                
                html.Div(style={'display': 'flex', 'gap': '30px'}, children=[
                    html.Div(style={'flex': '1'}, children=[
                        html.Label("Teatro de Operações:", style={'fontWeight': 'bold', 'display': 'block', 'marginBottom': '8px'}),
                        dcc.Dropdown(
                            id='filtro-theater',
                            options=[{'label': t, 'value': t} for t in lista_teatros],
                            value=lista_teatros[0],
                            clearable=False,
                            style=estilo_filtro
                        )
                    ]),
                    html.Div(style={'flex': '1'}, children=[
                        html.Label("Intensidade do Ataque (Carga de Bombas):", style={'fontWeight': 'bold', 'display': 'block', 'marginBottom': '8px'}),
                        dcc.Dropdown(
                            id='filtro-intensidade',
                            options=[{'label': i, 'value': i} for i in lista_intensidades],
                            value=[lista_intensidades[1], lista_intensidades[2]], 
                            multi=True,
                            style=estilo_filtro
                        )
                    ])
                ])
            ]),
            
            html.Div(style={'display': 'grid', 'gridTemplateColumns': '1fr', 'gap': '20px'}, children=[
                # O Mapa
                html.Div(style={'backgroundColor': bg_card, 'border': f'1px solid {borda_card}', 'padding': '15px', 'borderRadius': '8px'}, children=[
                    dcc.Graph(id='grafico-mapa-alvos')
                ]),

                # Ranking dos paises e Tipos de Aviões
                html.Div(style={'display': 'flex', 'gap': '20px'}, children=[
                    html.Div(style={'flex': '1', 'backgroundColor': bg_card, 'border': f'1px solid {borda_card}', 'padding': '15px', 'borderRadius': '8px'}, children=[
                        dcc.Graph(id='grafico-ranking-paises')
                    ]),
                    html.Div(style={'flex': '1', 'backgroundColor': bg_card, 'border': f'1px solid {borda_card}', 'padding': '15px', 'borderRadius': '8px'}, children=[
                        dcc.Graph(id='grafico-tipos-aeronaves')
                    ])
                ]),
                # Sazonalidade por mês e Altidade vs Carga
                html.Div(style={'display': 'flex', 'gap': '20px'}, children=[
                    html.Div(style={'flex': '1', 'backgroundColor': bg_card, 'border': f'1px solid {borda_card}', 'padding': '15px', 'borderRadius': '8px'}, children=[
                        dcc.Graph(id='grafico-sazonalidade-mes')
                    ]),
                    html.Div(style={'flex': '1', 'backgroundColor': bg_card, 'border': f'1px solid {borda_card}', 'padding': '15px', 'borderRadius': '8px'}, children=[
                        dcc.Graph(id='grafico-dispersao-altitude')
                    ])
                ])
            ])
        ])

# CALLBACK DO DASHBOARD

In [185]:
@app.callback(
    [Output('grafico-mapa-alvos', 'figure'),
     Output('grafico-ranking-paises', 'figure'),
     Output('grafico-tipos-aeronaves', 'figure'),
     Output('grafico-sazonalidade-mes', 'figure'),
     Output('grafico-dispersao-altitude', 'figure')],
    [Input('filtro-theater', 'value'),
     Input('filtro-intensidade', 'value'),
     Input('tema-toggle', 'value')] 
)
def atualizar_dashboard_exploratorio(teatro_sel, intensidade_sel, tema):
    template_grafico = 'plotly_dark' if tema == 'dark' else 'simple_white'
    
    df_filtrado = df_completo[df_completo['Teatro_Operacoes'] == teatro_sel]
    
    if not intensidade_sel or len(intensidade_sel) == 0:
        # Se o filtro estiver vazio, muda a tabela para zero linhas
        df_filtrado = df_filtrado.head(0) 
    else:
        # Só aplica a busca se tiver algo selecionado
        df_filtrado = df_filtrado[df_filtrado['Intensidade_Ataque'].isin(intensidade_sel)]

    if df_filtrado.empty:
        # Mapa Vazio com Globo 
        fig_mapa = px.scatter_geo(title=f'Mapeamento: {teatro_sel} (Nenhum alvo nos filtros atuais)', projection="natural earth")
        fig_mapa.update_layout(template=template_grafico, margin=dict(l=0, r=0, t=40, b=0))
        
        # Cria os outros 4 gráficos vazios com um aviso no meio
        def criar_grafico_vazio(titulo):
            fig = px.scatter(title=titulo)
            fig.update_layout(
                template=template_grafico,
                xaxis={'visible': False}, yaxis={'visible': False},
                annotations=[dict(
                    text="Nenhum registro encontrado para os filtros atuais.", 
                    xref="paper", yref="paper", x=0.5, y=0.5, showarrow=False, 
                    font=dict(size=14, color="gray")
                )]
            )
            return fig
            
        fig_barras = criar_grafico_vazio('Top 10 Países Alvo')
        fig_rosca = criar_grafico_vazio('Distribuição de Surtidas')
        fig_sazonalidade = criar_grafico_vazio('Sazonalidade Mensal')
        fig_dispersao = criar_grafico_vazio('Correlação Operacional: Altitude de Voo vs. Carga de Destruição')
        
        return fig_mapa, fig_barras, fig_rosca, fig_sazonalidade, fig_dispersao
    
    # Gráfico de Dispersão Espacial (Mapa de Alvos)
    df_mapa = df_filtrado.dropna(subset=['Latitude', 'Longitude'])
    df_mapa = df_mapa.sample(n=min(5000, len(df_mapa)), random_state=42) if len(df_mapa) > 0 else df_mapa
    
    mapa_de_cores = {'Leve': '#5c9664', 'Médio': '#d9a334', 'Pesado': '#c23b3b'}

    fig_mapa = px.scatter_geo(
        df_mapa, 
        lat='Latitude', 
        lon='Longitude', 
        size='Total_Toneladas',  # <-- ATUALIZADO
        color='Intensidade_Ataque',
        color_discrete_map=mapa_de_cores,
        title=f'Mapeamento Geográfico — Teatro {teatro_sel}', 
        projection="natural earth"
    )
    fig_mapa.update_layout(template=template_grafico, margin=dict(l=0, r=0, t=40, b=0))

    # Gráfico de Barras Horizontais (Top 10 Países Alvo)
    df_paises = df_filtrado.groupby('Pais_Alvo')['Total_Toneladas'].sum().reset_index().sort_values(by='Total_Toneladas', ascending=True).tail(10)
    fig_barras = px.bar(
        df_paises, x='Total_Toneladas', y='Pais_Alvo', orientation='h',
        title='Top 10 Países Alvo (por Tonelagem)',
        labels={'Total_Toneladas': 'Toneladas Lançadas', 'Pais_Alvo': ''}
    )
    fig_barras.update_layout(template=template_grafico)

    # Gráfico de Rosca (Distribuição de Aeronaves)
    df_aeronaves = df_filtrado['Categoria_Aeronave'].value_counts().reset_index().head(5)
    fig_rosca = px.pie(
        df_aeronaves, names='Categoria_Aeronave', values='count', hole=0.6,
        title='Distribuição de Surtidas por Tipo de Aeronave'
    )
    fig_rosca.update_traces(textposition='inside', textinfo='percent+label', showlegend=False)
    fig_rosca.update_layout(template=template_grafico)

    # Gráfico de Barras Verticais (Sazonalidade)
    df_mes = df_filtrado.groupby('Mes')['Total_Toneladas'].sum().reset_index()
    fig_sazonalidade = px.bar(
        df_mes, x='Mes', y='Total_Toneladas',
        title='Sazonalidade Mensal: Intensidade Logística de Bombardeios',
        labels={'Total_Toneladas': 'Toneladas de Bombas', 'Mes': 'Mês do Ano'}
    )
    fig_sazonalidade.update_layout(template=template_grafico)
    fig_sazonalidade.update_xaxes(tickmode='linear', dtick=1) 

    # Gráfico de Dispersão (Bolhas - Altitude vs Carga)
    df_dispersao = df_filtrado.dropna(subset=['altitude_feet', 'Total_Toneladas'])
    df_dispersao = df_dispersao.sample(n=min(2000, len(df_dispersao)), random_state=42) if len(df_dispersao) > 0 else df_dispersao
    
    fig_dispersao = px.scatter(
        df_dispersao, 
        x='altitude_feet', 
        y='Total_Toneladas', 
        color='Categoria_Aeronave',
        hover_name='Aeronave_Modelo',
        hover_data={'lore_historico': True, 'altitude_feet': True, 'Total_Toneladas': True, 'Categoria_Aeronave': False},
        title='Correlação Operacional: Altitude de Voo vs. Carga de Destruição',
        labels={'altitude_feet': 'Altitude (Pés)', 'Total_Toneladas': 'Toneladas', 'Categoria_Aeronave': 'Tipo'}
    )
    fig_dispersao.update_layout(template=template_grafico)

    return fig_mapa, fig_barras, fig_rosca, fig_sazonalidade, fig_dispersao

In [186]:
df_completo.head(6)

,ID_Missao,master_index_number,Data_Missao,Teatro_Operacoes,naf,Pais_Atacante,tgt_country_code,Pais_Alvo,Localizacao_Alvo,Tipo_Alvo,...,database_edit_comments,Sigla_Aeronave,Categoria_Aeronave,Aeronave_Modelo,Nome_Armamento,Classe_Armamento,lore_historico,Ano,Mes,Intensidade_Ataque
0,1,NaN,1943-08-15,Mediterrâneo (MTO),12 AF,USA,13.0,Itália,SPADAFORA,Não Identificado,...,NaN,A36,Ataque ao Solo / Bomb. de Mergulho,North American A-36 Apache (Invader),NaN,Não Identificado,O North American A-36 (designação de empresa N...,1943,8,Leve
1,4285,20028.0,1945-02-20,Pacífico (PTO),5 AF,USA,NaN,Filipinas,PUERTA PRINCESA,UNIDENTIFIED TARGET,...,NaN,A20,Bombardeiro Leve / Caça Noturno,Douglas A-20 Havoc,NaN,Não Identificado,O Douglas A-20 Havoc (designação de empresa DB...,1945,2,Leve
2,3,NaN,1943-08-15,Mediterrâneo (MTO),12 AF,USA,13.0,Itália,COSENZA,Não Identificado,...,NaN,A36,Ataque ao Solo / Bomb. de Mergulho,North American A-36 Apache (Invader),NaN,Não Identificado,O North American A-36 (designação de empresa N...,1943,8,Leve
3,4,NaN,1943-08-15,Mediterrâneo (MTO),12 AF,USA,13.0,Itália,GIOJA TAURO,Não Identificado,...,NaN,A36,Ataque ao Solo / Bomb. de Mergulho,North American A-36 Apache (Invader),NaN,Não Identificado,O North American A-36 (designação de empresa N...,1943,8,Leve
4,8167,14639.0,1945-02-23,Pacífico (PTO),5 AF,USA,NaN,Filipinas,BALETE PASS,WOODED AREA,...,NaN,A20,Bombardeiro Leve / Caça Noturno,Douglas A-20 Havoc,NaN,Não Identificado,O Douglas A-20 Havoc (designação de empresa DB...,1945,2,Leve
5,11286,20037.0,1945-02-26,Pacífico (PTO),5 AF,USA,NaN,Filipinas,PUERTA PRINCESA,UNIDENTIFIED TARGET,...,NaN,A20,Bombardeiro Leve / Caça Noturno,Douglas A-20 Havoc,NaN,Não Identificado,O Douglas A-20 Havoc (designação de empresa DB...,1945,2,Leve


In [187]:
if __name__ == '__main__':
    app.run(jupyter_mode="external", port=8052, debug=False)# ngrok http 8052(Se for usar ngrok por algum motivo)


Dash app running on http://127.0.0.1:8052/
